# 🆓 MONARCH STUDIO · Free-T4 video bridge (WF-06 · free route)

**Aap ka sawal:** "MiniMax-H3 free nahi hai?" — **sach: H3 ka API paid hai ($0.08–0.13/second).**
Yeh notebook H3 nahi chalata; yeh **woh chalata hai jo free Colab T4 (16 GB) par asal me chalta hai**:
**Wan 2.1 T2V 1.3B** (Apache-2.0, ~8 GB VRAM, 480p, 5 s) — yani **$0, bina API key**.

| | H3 (API) | Yeh notebook (free) |
|---|---|---|
| Paisa | $0.08–0.13 / second | **$0** (sirf Colab) |
| Quality | best-in-class, 2K | theek, 480p, 5 s |
| Audio | native 32 kHz stereo | **silent** → voice/music studio se add hoga |
| Ek clip ka waqt | 30 s – 4 min | **8–25 min** (T4 par) |
| 2K | ✅ | ❌ |

**Pattern wahi jo aap ke pichle agent ne diya tha:** notebook upload → Runtime → T4 GPU → Run all →
jobs banti jati hain, files Drive me. Neeche cell 8 me ek **optional tunnel** bhi hai (wahi "URL" system),
magar `TUNNEL_ENABLED = False` hai — pehle us cell ka warning parh lo.

### 3 zaroori sach (chhupa nahi raha)
1. **Colab free TOS:** free tier *interactive* use ke liye hai; 24/7 model server ya "remote control/SSH"
   allowed nahi. Tunnel chalu rakhna account ban ka risk hai — is liye default OFF hai.
2. **T4 purana GPU hai** (Turing, sm_75): FP8/NVFP4 nahi, `bfloat16` nahi → `float16`. Is liye
   Wan 2.2 / H3 / LTX-2.3 jaisi nayi cheezein T4 par practical nahi.
3. **Free me speed ki qeemat:** 5 s clip = 8–25 min. 10 jobs = poori raat. Session ~90 min idle par band,
   12 ghante me disk khatam — is liye **output Drive me** jata hai, local me nahi.

In [ ]:
#@title 1 · Hardware check (T4 expected)
import subprocess, sys, os, shutil, platform, json, pathlib, time

def sh(cmd, default=""):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=30).stdout.strip() or default
    except Exception:
        return default

print("Python :", sys.version.split()[0])
print("nvidia-smi:")
print(sh(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"]) or "  no GPU")
print("RAM    :", round(os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 2**30, 1), "GiB")
print("Disk   :", round(shutil.disk_usage('/').free / 2**30, 1), "GiB free")

GPU = sh(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"])
IS_T4 = "T4" in GPU
print("\nT4 detected:", IS_T4)
if not IS_T4:
    print("⚠️  T4 nahi mila. Runtime → Change runtime type → T4 GPU chuno.")
    print("    (CPU par yeh notebook na chalao — Wan 1.3B CPU par ghanton lega.)")
    print("    A100/L4 mile to bhi chalega (aur tez chalega).")

In [ ]:
#@title 2 · Install (torch Colab me pehle se hai)
!pip -q install "diffusers>=0.35" "transformers>=4.49" accelerate imageio imageio-ffmpeg ftfy sentencepiece safetensors

import torch, diffusers, transformers
print("torch      :", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.version.cuda)
print("diffusers  :", diffusers.__version__)
print("transformers:", transformers.__version__)
if torch.cuda.is_available():
    print("device     :", torch.cuda.get_device_name(0),
          "| VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1), "GiB")
    print("bf16 ok    :", torch.cuda.is_bf16_supported(), "(T4 par False hota hai → fp16 use karenge)")

In [ ]:
#@title 3 · Config
MODEL_ID   = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"   # Apache-2.0 · 480p · 81 frames @16fps ≈ 5 s
USE_DRIVE  = True                                  #@param {type:"boolean"}
DRIVE_SUB  = "MONARCH STUDIO/WF-06 · H3/free_t4"    #@param {type:"string"}
MAX_JOBS   = 8                                     #@param {type:"integer"}  <- ek session me kitne clips
STEPS      = 30                                    #@param {type:"integer"}  <- 20 tez, 30 behtar
GUIDANCE   = 5.0
FPS        = 16
RUN_ID     = time.strftime("t4-%Y%m%d-%H%M%S")

OUT_DIR = pathlib.Path("/content/free_t4") / RUN_ID
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = pathlib.Path("/content/drive/MyDrive") / DRIVE_SUB / RUN_ID
    except Exception as e:
        print("Drive mount skip:", e)
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Run  :", RUN_ID)
print("Out  :", OUT_DIR)

In [ ]:
#@title 4 · Load pipeline (fp16 + offload + tiling)
import torch, time
from diffusers import WanPipeline, AutoencoderKLWan

t0 = time.time()
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.float16)

vram = torch.cuda.get_device_properties(0).total_memory / 2**30
if vram < 20:
    pipe.enable_model_cpu_offload()      # T4 16 GB: layers CPU<->GPU shuffle
    pipe.enable_vae_tiling()
    try:
        pipe.enable_attention_slicing()
    except Exception:
        pass
else:
    pipe.to("cuda")
print(f"loaded in {time.time()-t0:.0f}s · mode={'offload' if vram < 20 else 'resident'} · {vram:.1f} GiB VRAM")

In [ ]:
#@title 5 · generate() — ek job = ek mp4 + receipt
import time, json, hashlib
from diffusers.utils import export_to_video

NEGATIVE = ("blurry, low quality, distorted, deformed hands, extra limbs, text overlay, watermark, "
            "jpeg artifacts, static frame, flicker")

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def generate(job, out_dir=OUT_DIR):
    w, h = job.get("width", 832), job.get("height", 480)
    frames = int(job.get("frames", 81))
    if (frames - 1) % 4:                      # Wan: num_frames must be 4k+1
        frames = ((frames - 1) // 4) * 4 + 1
    t0 = time.time()
    result = pipe(
        prompt=job["prompt"],
        negative_prompt=job.get("negative_prompt", NEGATIVE),
        height=h, width=w, num_frames=frames,
        num_inference_steps=int(job.get("steps", STEPS)),
        guidance_scale=float(job.get("guidance", GUIDANCE)),
        generator=torch.Generator(device="cpu").manual_seed(int(job.get("seed", 42))),
    )
    frames_out = result.frames[0]
    dest = out_dir / f"{job['id']}.mp4"
    export_to_video(frames_out, str(dest), fps=int(job.get("fps", FPS)))
    receipt = {"id": job["id"], "model": MODEL_ID, "prompt": job["prompt"],
               "width": w, "height": h, "frames": frames, "fps": int(job.get("fps", FPS)),
               "steps": int(job.get("steps", STEPS)), "seed": int(job.get("seed", 42)),
               "seconds_to_render": round(time.time() - t0, 1),
               "local_path": str(dest), "md5": md5(dest), "bytes": dest.stat().st_size,
               "audio": "none (add voice/music in WF-05)", "created_at": int(time.time())}
    print(f"  ✔ {dest.name} · {receipt['seconds_to_render']}s render · {receipt['bytes']/1e6:.1f} MB")
    return receipt

In [ ]:
#@title 6 · JOBS — yahan kaam likho (Monarch prompts)
# Silent clips hain: dialogue/music WF-05 me lagega. Composition/camera H3 grammar jaisa hi likho.
JOBS = [
    {"id": "free-01-monarch-closeup", "seed": 42, "frames": 81, "steps": 30,
     "prompt": ("Cinematic live-action close-up of a late-20s South Asian man with thick swept-back black "
                "hair, light beard, fitted black shirt and charcoal blazer, standing in a dark studio with a "
                "soft key light from upper-left and teal and amber rim lights. He looks straight into the lens, "
                "breathes once, and gives a slow confident smirk. The camera pushes in with small amplitude at "
                "slow speed. Shallow depth of field, film grain, 35mm lens.")},

    {"id": "free-02-noir-plate", "seed": 7, "frames": 81, "steps": 30,
     "prompt": ("Cinematic live-action noir, low-angle wide shot of a rain-wet cobblestone alley at night, "
                "one flickering amber neon sign, steam rising from a storm drain, a lone figure in a long "
                "charcoal coat walking away from the camera, their shadow stretching along the wet stones. "
                "The camera tracks the figure with small amplitude at slow speed. Cold teal shadows, warm "
                "amber highlights, heavy film grain.")},

    {"id": "free-03-reel-insert", "seed": 11, "frames": 81, "steps": 20,
     "prompt": ("3D CG render, macro product shot of a matte black object rotating slowly on a reflective "
                "obsidian plinth, a single hard key light sweeping across its machined edge, dust motes "
                "drifting in the beam, a teal gradient behind it. The camera arcs around the object with "
                "small amplitude at slow speed. The lower third of the frame stays empty dark space.")},
]

print(f"{len(JOBS)} job(s):", ", ".join(j["id"] for j in JOBS))


In [ ]:
#@title 7 · Run all (free — koi paisa nahi, sirf waqt)
RECEIPTS = []
started = time.time()
for i, job in enumerate(JOBS[:MAX_JOBS], 1):
    print(f"\n[{i}/{min(len(JOBS), MAX_JOBS)}] {job['id']}")
    try:
        RECEIPTS.append(generate(job))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print("  ✖ OOM — is job ke liye height/width/frames kam karo (e.g. 480x640, frames=49)")
        RECEIPTS.append({"id": job["id"], "error": "cuda OOM"})
    except Exception as e:
        print("  ✖", type(e).__name__, e)
        RECEIPTS.append({"id": job["id"], "error": f"{type(e).__name__}: {e}"})
    (OUT_DIR / "receipts.json").write_text(json.dumps(RECEIPTS, indent=2))

ok = [r for r in RECEIPTS if "error" not in r]
print(f"\nDone: {len(ok)}/{len(RECEIPTS)} clips in {round((time.time()-started)/60, 1)} min")
print("Receipts:", OUT_DIR / "receipts.json")
print("Drive/Colab folder se mp4 download karo, ya mujhe bolo — main Drive se utha lunga.")

## 8 · "Ping" style job queue (phone se jobs bhejo)

Yeh wahi idea hai jo aap ke pichle agent ne diya tha — magar **inbound tunnel ke baghair**:
Colab Drive ka `jobs.json` parhta rehta hai, naya job milta hai to bana ke `status.json` likh deta hai.
Aap phone se Drive me `jobs.json` edit karo → Colab utha lega. Band karo to kuch nahi bigadta.

In [ ]:
#@title 8 · Drive job queue (optional, safe version of the "ping" system)
QUEUE_ENABLED = False   #@param {type:"boolean"}
POLL_SECONDS  = 30      #@param {type:"integer"}
MAX_MINUTES   = 240     #@param {type:"integer"}  <- self-stop (Colab free ki mohlat)

import json, time, pathlib
JOB_FILE = OUT_DIR.parent / "jobs.json"
STATUS   = OUT_DIR.parent / "status.json"

if QUEUE_ENABLED:
    print("Queue dir:", JOB_FILE.parent)
    if not JOB_FILE.exists():
        JOB_FILE.write_text(json.dumps({"jobs": []}, indent=2))
        print("Template banaya:", JOB_FILE, "— Drive me edit karke jobs daalo")
    started, done = time.time(), set()
    while (time.time() - started) / 60 < MAX_MINUTES:
        try:
            data = json.loads(JOB_FILE.read_text())
        except Exception as e:
            print("jobs.json parh nahi paya:", e); time.sleep(POLL_SECONDS); continue
        pending = [j for j in data.get("jobs", []) if j.get("id") and j["id"] not in done]
        for job in pending:
            print("▶", job["id"])
            try:
                generate(job); done.add(job["id"]); status = "done"
            except Exception as e:
                done.add(job["id"]); status = f"error: {e}"
                print("  ✖", e)
            STATUS.write_text(json.dumps({j: "done" for j in done}, indent=2))
        time.sleep(POLL_SECONDS)
else:
    print("Queue off. (True karo to Drive-based job polling chalu hoga — unlimited jobs, one press.)")

## 9 · Tunnel / URL system (jaisa pichle agent ne kaha) — **par pehle yeh parh lo**

Pichla agent aap ko `colab_gpu_bridge_v2.ipynb` deta tha jo Colab par ek URL banata tha — matlab Colab ka GPU
ek chhota server ban jata hai aur bahar se koi bhi us URL par job bhej sakta hai.

**Sach:**
- Colab ke **free tier ki terms** "remote control / SSH / server" wale use ki ijazat nahi deti. URL chalu rakhna
  account suspension ka risk hai. (Paid Colab bhi server-hosting ke liye nahi hai; uske liye RunPod/Vast rent karo.)
- Quick tunnel URL **public** hota hai — token na ho to duniya bhar ka koi bhi aap ke GPU par job chala dega.
- Mera sandbox (jahan main chal raha hoon) sirf github/npm/pypi tak nikal sakta hai, **random tunnel URL tak nahi**.
  Is liye is session me main tunnel se job nahi bhej sakta — aap ke liye woh sirf browser/curl se kaam karega.
- Isi kaam ka **safe alternative**: upar cell 8 ka Drive queue (free, ToS-safe) — job file, output file.

Phir bhi chahiye to neeche cell hai; `TUNNEL_ENABLED = False` hai, token lazmi hai, aur 60 min par khud band.

In [ ]:
#@title 8b · Optional quick tunnel (TOS risk — off by default, token required)
TUNNEL_ENABLED = False                                     #@param {type:"boolean"}
API_TOKEN      = "CHANGE-ME-long-random-string"            #@param {type:"string"}
AUTO_STOP_MIN  = 60                                        #@param {type:"integer"}

import json, threading, subprocess, re, time, os, secrets
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

if TUNNEL_ENABLED:
    assert len(API_TOKEN) > 20 and API_TOKEN != "CHANGE-ME-long-random-string", "strong token set karo"
    print("⚠️  Colab free TOS: server-style use allowed nahi. Sirf apne test ke liye, band karna na bhoolein.")

    class Handler(BaseHTTPRequestHandler):
        def _send(self, code, obj):
            body = json.dumps(obj).encode()
            self.send_response(code); self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(body))); self.end_headers(); self.wfile.write(body)

        def do_GET(self):
            if self.headers.get("X-Token") != API_TOKEN:
                return self._send(401, {"error": "bad token"})
            self._send(200, {"ok": True, "model": MODEL_ID, "queue": str(OUT_DIR), "vram_gib": round(vram, 1)})

        def do_POST(self):
            if self.headers.get("X-Token") != API_TOKEN:
                return self._send(401, {"error": "bad token"})
            try:
                length = int(self.headers.get("Content-Length", 0))
                job = json.loads(self.rfile.read(length) or b"{}")
                job.setdefault("id", "api-" + secrets.token_hex(4))
                if len(job.get("prompt", "")) < 5:
                    return self._send(400, {"error": "prompt required"})
                receipt = generate(job)      # synchronous: response me hi file path aata hai
                return self._send(200, receipt)
            except Exception as e:
                return self._send(500, {"error": f"{type(e).__name__}: {e}"})

    server = ThreadingHTTPServer(("0.0.0.0", 8000), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()

    if not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run("wget -q -O /usr/local/bin/cloudflared "
                       "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
                       "&& chmod +x /usr/local/bin/cloudflared", shell=True, check=True)
    proc = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url = None
    for line in proc.stdout:
        m = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0); break
    print("\nPUBLIC URL:", url)
    print("Test:  curl -X POST %s -H 'X-Token: %s' -H 'Content-Type: application/json' "
          "-d '{\"id\":\"t1\",\"prompt\":\"a slow pan over a rainy street, cinematic\"}'" % (url, API_TOKEN))
    time.sleep(AUTO_STOP_MIN * 60)
    proc.terminate(); server.shutdown()
    print("tunnel stopped after", AUTO_STOP_MIN, "min")
else:
    print("Tunnel off (default). Cell 8 ka Drive queue use karo — free, aur TOS-safe.")

## 10 · Output ka kya karna hai

1. **Silent clips** Drive me: `MONARCH STUDIO/WF-06 · H3/free_t4/<run-id>/`.
2. Audio: (a) studio voice (`voice-00` style) se dialogue/VO, (b) music/SFX, (c) WF-05 me mix + typography.
3. **Hero shot** chahiye (asli dialogue + lipsync + 2K)? Woh H3 API se — 5 s 768P = $0.40, 2K = $0.65.
   Free Wan clip ko draft/plate ki tarah use karo, hero H3 se banao: paisa sirf jahan quality dikhti hai.
4. Batch khatam hone par mujhe bolo — main Drive se `receipts.json` + mp4 utha kar gates (format, look,
   continuity) chala kar WF-05 ke liye trim/caption plan bana dunga.

### Free vs paid — ek line me
**Free me sab kuch milega magar (480p, silent, 10–25 min/clip); paise se (2K, native stereo audio, 30 s/clip).**
Dono ko mila kar chalao: plates free, hero paid.

In [ ]:
#@title 11 · Session summary
print("Run      :", RUN_ID)
print("Folder   :", OUT_DIR)
clips = sorted(OUT_DIR.glob("*.mp4"))
print("Clips    :", len(clips))
for c in clips:
    print(f"  {c.name:<34} {c.stat().st_size/1e6:6.1f} MB  {md5(c)[:10]}")
print("\nAgla step: mujhe 'clips ready' bolo — main Drive se utha kar gates + WF-05 plan bana dunga.")